# PixiJoy Monocular depth (M2 / H5) — Colab GPU

Train → TFLite → contract verify. **Runtime → Change runtime type → GPU** then **Runtime → Run all**.

When finished, download `dist/depth.tflite` and attach to [Release- ml-packs-v1](https://github.com/Saikumar-S0906/Release-/releases/tag/ml-packs-v1).


In [ ]:
!nvidia-smi || true
import tensorflow as tf
print("TF", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))


In [ ]:
import os, urllib.request, zipfile
from pathlib import Path
from google.colab import files

ROOT = Path("/content/pixijoy_depth")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

ZIP_URL = "https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/depth_lab.zip"
ok = False
try:
    urllib.request.urlretrieve(ZIP_URL, "depth_lab.zip")
    with zipfile.ZipFile("depth_lab.zip") as z:
        z.extractall(".")
    ok = Path("train.py").is_file()
    print("extracted lab from Release- main", ok)
except Exception as e:
    print("zip fetch failed:", e)

if not ok:
    print("Upload depth_lab.zip (from tools/depth/ after pack_ml_labs.ps1)")
    uploaded = files.upload()
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall(".")
    ok = Path("train.py").is_file()
assert ok, "Could not load lab zip"


In [ ]:
!python -u train.py
!python -u export_tflite.py
!python -u verify_contract.py
from pathlib import Path
import hashlib
p = Path("dist/depth.tflite")
print("READY", p, p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())


In [ ]:
from google.colab import files
files.download("dist/depth.tflite")
print("Upload this file to GitHub Release ml-packs-v1 as depth.tflite")
